In [ ]:







from devices_new.PMDevice import PMDevicePM100D


In [ ]:
#Подключение РЧ анализатора
rf_device=RF306B()
rf_device.set_refLevel(-20)
frequencies, powers, peak_freq, peak_power = measure_rf_spectrum(
    rf_device,
    f_start=9000, # 9 kHz start
    f_stop=6e9   # 6 GHz stop
)
plt.plot(frequencies, powers)

In [ ]:

scan_modes=get_hysteresis_sequences(currents=currents,delays=delays,mode='both')
for scan in scan_modes:
    currents=scan[0]
    delays=scan[1]
    scan_name=scan[2]
    print('scan_name =',scan_name)

In [ ]:

main_DATA_folder = create_date_folder()

for scan in scan_modes:
    currents=scan[0]
    delays=scan[1]
    scan_name=scan[2]
    print('scan_name =',scan_name)

    for voltage in voltages:
        ut.set_voltage(voltage)
        
        power_data_max_span, freq_data_max_span = [], []
        power_data_middle_span, freq_data_middle_span = [], []
        power_data_min_span, freq_data_min_span = [], []
        pm_power_data = []
        currents_data, delays_data = [], []

        
        voltage_folder=create_subfolder(main_DATA_folder,"voltage_"+str(voltage))
        scan_folder=create_subfolder(voltage_folder,scan_name)
        
        max_span_name=format_number_with_prefix(max_span)
        middle_span_name=format_number_with_prefix(middle_span)
        min_span_name=format_number_with_prefix(min_span)
        
        max_span_folder=create_subfolder(scan_folder,f'span_{max_span_name}Hz')
        middle_span_folder=create_subfolder(scan_folder,f'span_{middle_span_name}Hz')
        min_span_folder=create_subfolder(scan_folder,f'span_{min_span_name}Hz')
        
        
        for current in currents:
           
            LD.set_current(current)
            time.sleep(t_delay)
        
            for delay in delays:
                print(f'I = {current}, delay = {delay}' )
                
                odl.set_time_delay(delay)
                time.sleep(t_delay)
            
                real_current=LD.get_current()
                real_delay=odl.get_time_delay()

                
                currents_data.append(real_current)
                delays_data.append(real_delay)

                pm_power = measure_average_power(pm_device,power_meter_coefficient)
                pm_power_data.append(pm_power)
        
                txt_filename=f'current_{current}_delay_{delay}_'
                avg_peak_freq_max_span, avg_peak_power_max_span=rf_measurement(rf_device,N,max_span_folder,txt_filename,
                                                                               rbw_max_span,span=None,f_center=None,
                                                                               f_start=f_start_max_span,f_stop=f_stop_max_span)
    
                power_data_max_span.append(avg_peak_power_max_span)  
                freq_data_max_span.append(avg_peak_freq_max_span)    
        
               
                f_center_middle_span = avg_peak_freq_max_span
                avg_peak_freq_middle_span, avg_peak_power_middle_span=rf_measurement(rf_device,N,middle_span_folder,txt_filename,
                                                                                     rbw_middle_span,middle_span,f_center_middle_span)
                power_data_middle_span.append(avg_peak_power_middle_span)
                freq_data_middle_span.append(avg_peak_freq_middle_span)
                
                
                
                
                f_center_min_span = avg_peak_freq_middle_span
                avg_peak_freq_min_span, avg_peak_power_min_span=rf_measurement(rf_device,N,min_span_folder,txt_filename,rbw_min_span,
                                                                               min_span,f_center_min_span)
                power_data_min_span.append(avg_peak_power_min_span)
                freq_data_min_span.append(avg_peak_freq_min_span)


        current_label='current,mA'
        delay_label='delay, ps'
        pm_power_label='power meter, mW'
        freq_label='freq, Hz'
        rsa_power_label='power, dBm'
        # сохраняем и отрисовываем карты   
        create_map_and_save(currents_data,delays_data,pm_power_data,current_label,delay_label,pm_power_label,scan_folder,'pm_power_data')
        
        create_map_and_save(currents_data,delays_data,freq_data_max_span,current_label,delay_label, freq_label,scan_folder,'freq_data_max_span')
        
        create_map_and_save(currents_data,delays_data,freq_data_middle_span,current_label,delay_label, freq_label,scan_folder,'freq_data_middle_span')     
        
        create_map_and_save(currents_data,delays_data,freq_data_min_span,current_label,delay_label, freq_label,scan_folder,'freq_data_min_span') 
        
        create_map_and_save(currents_data,delays_data,power_data_max_span,current_label,delay_label,rsa_power_label,scan_folder,'power_data_max_span')
        
        create_map_and_save(currents_data,delays_data,power_data_middle_span,current_label,delay_label,rsa_power_label,scan_folder,'power_data_middle_span')
        
        create_map_and_save(currents_data,delays_data,power_data_min_span,current_label,delay_label,rsa_power_label,scan_folder,'power_data_min_span')
            

ut.set_voltage(0)
LD.set_current(0)
odl.set_time_delay(0)

LD.turn_off_laser()
LD.turn_off_tec()
odl.disconnect()

